## Build Results Fact

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01-Environment_Config

In [0]:
%run ../00-common/04-Gold_Helpers

In [0]:
results_table = f"{catalog_name}.{silver_schema}.results"
sprints_table = f"{catalog_name}.{silver_schema}.sprints"

fact_session_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as f

In [0]:
results_df = spark.read.table(results_table)\
                        .filter((f.col("batch_id") == v_batch_id))\
                            .withColumn("session_type", f.lit('RACE'))\
                                .drop("race_name", "race_date", "ingestion_timestamp", "source_file", "batch_id", "created_timestamp", "updated_timestamp")

sprints_df = spark.read.table(sprints_table)\
                        .filter((f.col("batch_id") == v_batch_id))\
                            .withColumn("session_type", f.lit('SPRINT'))\
                                .drop("race_name", "race_date", "ingestion_timestamp", "source_file", "batch_id", "created_timestamp", "updated_timestamp")

In [0]:
complete_df = results_df.unionByName(sprints_df)

In [0]:
fact_session_results_df =  (
    complete_df.withColumn("is_win", f.col("finish_position") == 1)\
                .withColumn("is_podium", f.col("finish_position").between(1, 3))\
                    .withColumn("has_points", f.col("points") > 0)
)

In [0]:
columns = [
    "grid_position",
    "completed_laps",
    "car_number",
    "points",
    "finish_position",
    "finish_position_text",
    "status",
    "is_win",
    "is_podium",
    "has_points"
]

write_to_gold(fact_session_results_df, fact_session_table, "trg.season = src.season AND trg.round = src.round AND trg.constructor_id = src.constructor_id AND trg.driver_id = src.driver_id AND trg.session_type = src.session_type", columns)